# AutoMorphClass on three shapes, biomarker by biomarker

A close reading of one implementation. The
[main notebook](../biomarker-synthetic.ipynb) compares six implementations across thirteen shapes and answers
*which of these agree*; this one asks a narrower question about
[AutoMorphClass](../../../docs/projects/automorphclass.md) alone — **what does it actually return, on the three
simplest shapes, at each angle, beside what the geometry requires?**

Everything is under **AutoMorphClass's own column names**, not the catalogued ones. That is the point:
the catalogued names exist to make two programs comparable, and there is only one program here, so
the names its authors chose are the ones a reader checking against their documentation will look
for. Where a column maps onto a catalogued biomarker the mapping is shown beside it, and where the
shape settles a theoretical value that is shown too.

**The three shapes, in order of how much they ask for:**

| Shape | What it is | Why it comes first |
| --- | --- | --- |
| `straight` | one straight vessel per class, leaving the optic disc | every curvature measure must read zero and every tortuosity exactly 1 |
| `arc` | a circular arc per class | curvature is exactly 1/r, and the arc-to-chord ratio has a closed form |
| `sinusoid` | a sine wave per class | the only one of the three whose curvature changes sign |

A blank in the theory column means **this shape settles no value for that quantity** — which is not
the same as the quantity being wrong, and not the same as AutoMorphClass declining to answer it.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def repository() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(f"nothing above {Path.cwd()} looks like the fundus-atlas repository")


ROOT = repository()
sys.path.insert(0, str(ROOT / "src"))

from benchmarks import biomarker_synthetic as benchmark  # noqa: E402
from benchmarks.shapes import store  # noqa: E402
from biomarkers import canonical  # noqa: E402
from biomarkers.utils import catalogue  # noqa: E402

#: The one implementation this notebook reads. Change it and everything below follows.
IMPLEMENTATION = "automorphclass"

#: In order of how much they ask for: a straight vessel, then a curve, then a curve that changes
#: its mind about which way it bends.
SHAPES = ("straight", "arc", "sinusoid")

EVIDENCE = ROOT / "results" / benchmark.NAME / IMPLEMENTATION
STORE = ROOT / store.STORE
pd.set_option("display.width", 200, "display.max_rows", 200)

#: AutoMorph's own column name → the catalogued biomarker it answers to, or None where the
#: catalogue has no name for it. This is the adapter's claim, not the run's.
NAMES = catalogue.load(IMPLEMENTATION).declare()["names"]

TRUTH = pd.read_csv(STORE / "ground_truth.csv").set_index("key")

#: How many microns a pixel is worth, read from the manifest rather than assumed.
UM_PER_PX = float(pd.DataFrame(store.read(STORE))["um_per_px"].iloc[0])
print(f"{IMPLEMENTATION}: {len(NAMES)} columns, "
      f"{sum(1 for v in NAMES.values() if v)} of them with a catalogued name")

automorphclass: 60 columns, 33 of them with a catalogued name


In [2]:
def readings(shape: str) -> pd.DataFrame:
    """What AutoMorph returned for one shape: a row per column, a column per angle.

    The theoretical value is joined on last and only where the shape settles one, so a blank there
    **Every number in a row is in the implementation's own units, which are pixels.** A catalogued
    biomarker is in microns, so the theory is converted *back* onto the pixel grid before it is put
    beside figures this program actually printed. The `unit` column says what the catalogued name
    is in.

    The theoretical value is joined on last, so a blank is a statement about the fixture rather
    than about AutoMorph.
    """
    measured = pd.read_csv(EVIDENCE / f"{shape}.csv")
    rows = {}
    for own in NAMES:
        values = {}
        for _, rendering in measured.iterrows():
            values[f"{rendering['rotation']:.0f}°"] = rendering.get(f"said_{own}")
        catalogued = NAMES[own]
        truth, unit = np.nan, ""
        if catalogued:
            unit = canonical.unit(catalogued)
            keys = [k for k in TRUTH.index if k.startswith(f"{shape}-")]
            if keys and catalogued in TRUTH.columns:
                # Back onto the pixel grid: the inverse of the one conversion this repository makes.
                truth = canonical.from_pixels(
                    catalogued, TRUTH.loc[keys[0], catalogued], 1.0 / UM_PER_PX
                )
        rows[own] = {
            **values,
            "theory, in px": truth,
            "catalogued as": catalogued or "—",
            "unit": unit or "—",
        }
    frame = pd.DataFrame(rows).T
    frame.index.name = f"{IMPLEMENTATION}'s own name"
    return frame


def show(shape: str) -> pd.DataFrame:
    frame = readings(shape)
    angles = [c for c in frame.columns if c.endswith("°")]
    numeric = frame[angles + ["theory, in px"]].astype(float)
    spread = numeric[angles].max(axis=1) - numeric[angles].min(axis=1)
    out = numeric.round(4)
    out["moves"] = spread.round(4)
    out["catalogued as"] = frame["catalogued as"]
    out["unit"] = frame["unit"]
    return out

## 1. `straight`

A single straight vessel per class, leaving the optic disc. **Everything about it is known
exactly**: the arc-to-chord ratio is 1, every curvature measure is 0, and the width is the width it
was drawn at. Nothing here is a hard measurement, which is what makes it the right place to start —
a value that is wrong on a straight line is wrong for a reason that has nothing to do with
difficulty.

In [3]:
show("straight")

,0°,15°,30°,45°,60°,75°,90°,"theory, in px",moves,catalogued as,unit
automorphclass's own name,,,,,,,,,,,
vessel_density_vessels,0.0048,0.0049,0.0049,0.0046,0.0049,0.0048,0.0048,0.0049,0.0002,density/over-image/vessels,1
fractal_dimension_vessels,0.9329,0.9998,0.9275,0.8923,0.9847,0.9782,0.9884,1.0000,0.1074,density/box-counting/vessels,1
average_width_vessels,16.0344,16.6019,18.5240,28.1101,18.4341,16.5928,15.9776,15.8025,12.1325,calibre/width/vessels/length-weighted,µm
distance_tortuosity_vessels,1.0000,1.0743,1.0731,1.0104,1.0751,1.0732,1.0000,1.0000,0.0751,tortuosity/hart-tau1/vessels/length-weighted,1
squared_curvature_tortuosity_vessels,0.0000,10.5311,3.9728,1.1677,5.6067,9.8063,0.0000,NaN,10.5311,—,—
tortuosity_density_vessels,0.0000,0.0193,0.0069,0.0025,0.0119,0.0106,0.0000,0.0000,0.0193,tortuosity/grisan-density/vessels/length-weighted,1/µm
vessel_density_artery,0.0017,0.0019,0.0019,0.0017,0.0019,0.0019,0.0017,0.0019,0.0002,density/over-image/artery,1
fractal_dimension_artery,0.9281,0.9245,0.8537,0.8624,0.8411,0.9282,0.9329,1.0000,0.0918,density/box-counting/artery,1
average_width_artery,11.3550,13.2269,14.7844,15.4481,14.7844,13.2269,11.3953,12.6420,4.0931,calibre/width/artery/length-weighted,µm


## 2. `arc`

A circular arc per class, the two sharing a subtended angle and differing in radius. The
arc-to-chord ratio is `θ / (2 sin(θ/2))` in closed form and **independent of radius**, so both
classes must return the same tortuosity while their curvatures differ. That pairing is the test:
an implementation that returns different tortuosities for the two is measuring the radius somewhere
it should not.

In [4]:
show("arc")

,0°,15°,30°,45°,60°,75°,90°,"theory, in px",moves,catalogued as,unit
automorphclass's own name,,,,,,,,,,,
vessel_density_vessels,0.0050,0.0051,0.0050,0.0050,0.0051,0.0050,0.0050,0.0050,0.0000,density/over-image/vessels,1
fractal_dimension_vessels,1.0808,1.1032,1.0549,1.0555,1.0700,1.0602,1.0797,1.0000,0.0483,density/box-counting/vessels,1
average_width_vessels,17.6863,17.4793,16.5604,16.7919,17.5025,17.9564,17.6948,15.3510,1.3959,calibre/width/vessels/length-weighted,µm
distance_tortuosity_vessels,1.1325,1.1153,1.1296,1.1244,1.1169,1.1284,1.1277,1.0650,0.0172,tortuosity/hart-tau1/vessels/length-weighted,1
squared_curvature_tortuosity_vessels,5.0114,5.0056,6.7108,6.6500,5.5718,6.1024,6.2461,NaN,1.7052,—,—
tortuosity_density_vessels,0.0091,0.0086,0.0110,0.0102,0.0090,0.0114,0.0094,0.0000,0.0028,tortuosity/grisan-density/vessels/length-weighted,1/µm
vessel_density_artery,0.0024,0.0024,0.0023,0.0023,0.0024,0.0024,0.0024,0.0024,0.0000,density/over-image/artery,1
fractal_dimension_artery,0.9752,0.9580,0.9091,0.8969,0.9289,0.9489,0.9834,1.0000,0.0865,density/box-counting/artery,1
average_width_artery,14.4414,14.2198,13.5014,13.6919,14.3413,14.5370,14.3543,12.6420,1.0356,calibre/width/artery/length-weighted,µm


## 3. `sinusoid`

A sine wave per class, and **the only one of the three whose curvature changes sign**. Its arc
length is an elliptic integral with no elementary closed form, so the theoretical value here is the
integral rather than a formula. It is the shape that settles the inflection count and Grisan's
density, both of which are zero on everything else.

In [5]:
show("sinusoid")

,0°,15°,30°,45°,60°,75°,90°,"theory, in px",moves,catalogued as,unit
automorphclass's own name,,,,,,,,,,,
vessel_density_vessels,0.0055,0.0055,0.0056,0.0055,0.0055,0.0055,0.0055,0.0055,0.0000,density/over-image/vessels,1
fractal_dimension_vessels,1.0187,1.0262,0.9643,0.9373,0.9265,1.0170,1.0183,1.0000,0.0997,density/box-counting/vessels,1
average_width_vessels,18.4284,18.1744,17.8144,17.3817,17.4642,17.9260,18.5394,15.8025,1.1576,calibre/width/vessels/length-weighted,µm
distance_tortuosity_vessels,1.2017,1.2049,1.1980,1.2004,1.1977,1.1983,1.1982,1.1398,0.0072,tortuosity/hart-tau1/vessels/length-weighted,1
squared_curvature_tortuosity_vessels,5.7216,5.8421,5.7836,6.6404,6.1372,5.9259,5.4172,NaN,1.2231,—,—
tortuosity_density_vessels,0.0100,0.0094,0.0092,0.0084,0.0090,0.0092,0.0095,0.0006,0.0015,tortuosity/grisan-density/vessels/length-weighted,1/µm
vessel_density_artery,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0000,density/over-image/artery,1
fractal_dimension_artery,0.9714,0.9768,0.8494,0.8474,0.8485,0.9903,0.9669,1.0000,0.1429,density/box-counting/artery,1
average_width_artery,14.7000,14.3562,13.9989,13.7912,13.9456,14.4428,14.7523,12.6420,0.9611,calibre/width/artery/length-weighted,µm


## 4. What the three tables show

*Written in September 2026 against an earlier drawing of the shapes, at four angles. The tables
above were re-executed on 2026-11-25 against the store the benchmark now uses — seven angles,
6.33 µm a pixel — and they no longer show everything this section first described. Each subsection
keeps the earlier finding and says, under **Now**, what the current tables show. This notebook is
not of record ([why](../../wip/README.md)); the reading of record is the
[AutoMorph group notebook](../biomarker-synthetic-automorph.ipynb), §3.3 and §9.*

AutoMorphClass is a rewrite of [AutoMorph](biomarker-synthetic-automorph.ipynb)'s measuring stage,
and the most useful way to read these tables is against that one.

### 4.1 Every reading is a possible value, which is not true of its ancestor

AutoMorph returns **exactly 0.0000** for `distance_tortuosity` on `straight` at 30° and 60°, and on
`sinusoid` at 0° and 90°. An arc over a chord cannot be less than 1, so those are failures reported
as numbers. **Nothing in these three tables is impossible**: every tortuosity is at or above 1,
every width is positive, every fractal dimension is between 0 and 2. Whatever else follows, the
rewrite fixed the thing that most needed fixing, because a wrong number you can see is worth more
than a failure you cannot.

**Now:** still true of AutoMorphClass — every tortuosity in the current tables is at or above 1. Its
ancestor no longer returns 0 on these shapes; it now returns impossibly large values instead (166.8
for the vein on `arc` at 45°, 118.0 for the artery on `sinusoid` at 0°), so the contrast stands.

### 4.2 Steady, and biased high

On `arc`, where the geometry requires **1.1107** at every angle and for both classes:

| | 0° | 30° | 60° | 90° | theory | moves |
| --- | --- | --- | --- | --- | --- | --- |
| `distance_tortuosity_artery` | 1.1697 | 1.1704 | 1.1715 | 1.1739 | 1.1107 | 0.0042 |
| `distance_tortuosity_vein` | 1.1703 | 1.1722 | 1.1723 | 1.1704 | 1.1107 | 0.0020 |

Turning the picture moves it by **0.4%** — and it is **5.3% too high at every angle**. The two
classes agree with each other to three decimals, which is what the geometry requires of them, and
both agree on the wrong value. That is the opposite failure from AutoMorph, whose vein reads 1.037,
1.062, 7.133 and 5.367 on the same shape: a steady bias can be corrected once it is known, and
noise of that size cannot be corrected at all.

The same holds on `sinusoid` — 1.43 to 1.46 against a required 1.3726, about 5% high and moving by
2% — and on `straight`, where it reads 1.0007 at the axis-aligned angles and 1.073 at the diagonal
ones.

**Now:** the arc has been redrawn and requires **1.0650**. AutoMorphClass's length-weighted arc over
chord reads 1.113–1.131 for the artery and 1.117–1.137 for the vein across the seven angles: still
steady and still high, by 4.5–6.7%. On `sinusoid` it reads 1.191–1.206 against 1.1398, and on
`straight` exactly 1.0000 at 0° and 90° and 1.002–1.078 in between. AutoMorph's vein on `arc` now
reads 1.150, 24.8, 104.5, 166.8, 25.9, 1.43 and 1.126.

### 4.3 The width error runs the opposite way to AutoMorph's

On `straight`, one unchanging vessel at four angles:

| | 0° | 30° | 60° | 90° | drawn at |
| --- | --- | --- | --- | --- | --- |
| `average_width_vein`, **AutoMorphClass** | 24.87 | 28.52 | 28.41 | 24.87 | 24 |
| `average_width_vein`, **AutoMorph** | 14.15 | 24.94 | 25.13 | 14.15 | 24 |

Both wobble with the angle and they wobble **in opposite directions**. AutoMorphClass is nearly
exact where the vessel is axis-aligned and 18% wide where it is diagonal; AutoMorph is nearly exact
where it is diagonal and 41% narrow where it is axis-aligned. One of the two rewrote the width
calculation and the tables cannot say which is nearer right in general — only that on this shape
the rewrite is closer, and that the two cannot both be measuring what they claim to measure.

**Now:** superseded. The adapters now hand AutoMorph and AutoMorphClass the same 912 × 912 maps, and
the two return **identical** whole-picture widths: `average_width_vein` on `straight` reads 20.76 px
at 0°, 22.26 at 30°, 53.81 at 45°, 22.04 at 60° and 20.54 at 90° in both, against 18.96 required.
Whatever separated them when this was written, it is not in the widths they compute now.

### 4.4 A straight line with a fractal dimension of 0.376

`fractal_dimension_artery` on `straight` reads **0.3758, 0.8849, 0.8849, 0.3758**. A straight line
has dimension 1, and a bounded raster of one will read a little under it — 0.88 is a plausible
underestimate and 0.376 is not a dimension of anything.

**AutoMorph does not do this.** Its `fractal_dimension_artery` on the same shape reads 0.927,
0.977, 0.977, 0.927. So this is something the rewrite introduced rather than something it
inherited, and it appears only at the two axis-aligned angles.

**Now:** no longer seen. `fractal_dimension_artery` on `straight` reads 0.84–0.93 at every angle,
exactly AutoMorph's values, for the reason given in §4.3.

### 4.5 What is not checked here

Every fractal dimension, both per-class vessel densities and `squared_curvature_tortuosity` carried
no theory on these shapes when this was written (**now** 27 of its 60 columns carry none: the
`squared_curvature_tortuosity` columns, the zone vessel densities and the zone-C columns, none of
which has a catalogued name) — the first two because these shapes settle no value for them, the last
because the catalogue has no name for it at all. Those rows are reported and unchecked, which is a
third state and not a pass.
